# Chapter 4 (part 4) — Regression Diagnostics

Covers the Ch 4 section on testing the assumptions: Outliers → Influential Values → Heteroskedasticity, Non-Normality, and Correlated Errors → Partial Residual Plots and Nonlinearity, on Ames Housing.

The thread: **every regression result comes with a list of promises (linear mean, independent errors, constant spread, ...) and the diagnostics are how you check which promises were broken and, more importantly, what each broken promise costs you.** Some violations bias the coefficients, some only wreck the standard errors, some only matter for prediction intervals. Knowing which is which is exactly the "senior data scientist" intuition.

A note on the book's examples: the book's correlated-errors example is about time-ordered residuals and the Durbin-Watson statistic. The rows in this dataset come sorted roughly by neighborhood, not by date, so you'll build the time ordering yourself from `Year_Sold`/`Mo_Sold` and see what the raw file order does to the statistic. A spatial extension using `Latitude`/`Longitude` is included and labeled as supplementary (not in the book).

Base model for the exercises: `Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF`. (The exam uses a *different* model so it's not an answer key.)

Ask for a hint if you get stuck; the first couple of asks per question get a nudge, not the answer.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
base = smf.ols('Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF', ames).fit()
base.summary().tables[1]

## 1. Outliers

An **outlier** in a regression is a house whose price is far from what the model predicts, i.e. a big *residual*. Since residuals naturally scale with the data, we standardize them: divide each by its estimated standard deviation, so "3" means "three standard deviations of typical miss". (A *studentized* residual refits the model without that point first, so the point can't hide by pulling the line toward itself — statsmodels calls these `resid_studentized_external`.) Numeric picture: if typical misses are about $41k, a house missing by $123k has a standardized residual of 3.

**Question 1.** (a) Using `OLSInfluence(base)`, get externally studentized residuals. How many homes exceed |3|? |4|? What fraction is that, and how does it compare to what you'd expect if the residuals were normal? (b) Look at the 10 largest in absolute value: print `Neighborhood`, `Sale_Condition`, `Gr_Liv_Area`, `Sale_Price`, fitted price. Is there a story? (c) Are the large residuals mostly *over-* or *under-*predictions? (d) Is a big residual a data error, a special sale, or a model deficiency? How would you tell the three apart?

*Tools for this question:*
- `infl = OLSInfluence(res)` (from `statsmodels.stats.outliers_influence`); `infl.resid_studentized_external` is the studentized-residual array. Count with `(np.abs(x) > 3).sum()`.

_Your interpretation:_

## 2. Influential Values

A point has **influence** if deleting it changes the fitted coefficients. Influence needs *both* ingredients: a point far from the others in x-space (**leverage**, the *hat value*) *and* a miss from the line. Numeric picture: balance a ruler on a pencil. A weight near the middle barely tilts it (low leverage); a weight at the far end tips it hard (high leverage) — but a far-end weight that sits right on the line the ruler already follows doesn't tilt it at all (leverage without a big residual). **Cook's distance** combines both into one number: roughly "how far all the fitted values move if this point is deleted."

**Question 2.** (a) Compute hat values and Cook's D from `OLSInfluence`. Flag leverage > 2p/n and Cook's D > 4/n (p = number of parameters). How many are flagged by each? (b) Draw a leverage-vs-studentized-residual plot (`sm.graphics.influence_plot(base)` or build it yourself). Find houses in each corner: high leverage + big residual, high leverage + small residual, low leverage + big residual. List a couple of each. (c) Refit with the 5 highest-Cook's-D homes removed. By how much does each coefficient move in units of its standard error? (`DFBETAS` do this per point; you can compute it by hand from the two fits.) (d) The five homes with `Gr_Liv_Area > 4000` — print them. Do you remove them? Name a justification you'd be comfortable defending and one you wouldn't.

*Tools for this question:*
- `infl.hat_matrix_diag` leverage; `infl.cooks_distance` returns a tuple `(distances, p_values)` — take `[0]`; `infl.dfbetas` is an n × p array of per-coefficient influence.
- `sm.graphics.influence_plot(res, criterion="cooks")`; `df.drop(index=idx_list)` to refit without rows.

_Your interpretation:_

## 3. Heteroskedasticity

OLS assumes the spread of errors is the same at every x. With houses, that's rarely true: a $100k house might miss by $15k, a $600k house by $80k. If spread varies, the coefficients stay **unbiased** (the line is still centered correctly on average), but the standard formula for standard errors assumes the wrong variance, so p-values and confidence intervals are unreliable — in either direction.

**Question 3.** (a) Plot residuals against fitted values. Compute residual SD within fitted-value quintiles. Describe the shape. (b) Run `het_breuschpagan` on the residuals. With n = 2,930, what does a tiny p-value really tell you, and why should you look at the *size* of the heteroskedasticity too? (c) Re-fit with `HC3` robust standard errors (`.fit(cov_type='HC3')`). Compare the coefficients (identical?) and standard errors (how much change, for which variables?). (d) Re-fit on `np.log(Sale_Price)`. Re-check the residual spread by fitted quintile. Did the log fix it? Interpret the `Year_Built` coefficient in this model in percent terms — what does 0.01 mean? (e) Which fix would you pick if the question is "what is the effect of a year of age on price?" vs. "predict dollars for a new listing"? (Think: the log model predicts the *median* on the dollar scale unless corrected.)

*Tools for this question:*
- `het_breuschpagan(res.resid, res.model.exog)` (from `statsmodels.stats.diagnostic`) returns `(LM stat, LM p-value, F stat, F p-value)`.
- Robust SEs: `smf.ols(...).fit(cov_type="HC3")`. Quintiles: `pd.qcut(res.fittedvalues, 5, labels=False)` then `groupby(...)["resid"].std()`.

_Your interpretation:_

## 4. Non-Normality

Normal errors are the *least* important assumption for the coefficients when n is large, because coefficient estimates are weighted sums of the data, and sums become approximately normal by the central limit theorem even when the individual errors aren't. Non-normality matters mostly for **prediction intervals**, which are about one specific future error rather than an average.

**Question 4.** (a) Make a QQ plot of the standardized residuals (`sm.qqplot(..., line='45')`). Where do the tails depart, and in which direction — heavy right, heavy left, both? Report skewness and excess kurtosis. (b) Build a bootstrap CI (1,000 resamples) for the `Gr_Liv_Area` coefficient and compare to the classical 95% CI. Are they close? Why would you expect that? (c) Hold out 30% of the data. Build classical 95% prediction intervals for the held-out homes (`get_prediction(...).summary_frame()`), and compute the fraction of held-out prices that actually fall inside. Do this overall and within three tiers of predicted price. What do you find? (d) In one or two sentences: "what's the consequence of non-normal errors, depending on whether I'm doing inference on a coefficient or building a prediction interval?" 

*Tools for this question:*
- `sm.qqplot(x, line="45", fit=True)` (standardizes `x` first); `scipy.stats.skew(x)`, `scipy.stats.kurtosis(x)` (excess kurtosis; normal = 0).
- `res.get_prediction(new_df).summary_frame(alpha=0.05)` has columns `mean`, `mean_ci_lower/upper` (for the average) and `obs_ci_lower/upper` (prediction interval for one new house). `res.conf_int()` gives coefficient CIs.

_Your interpretation:_

## 5. Correlated Errors

If one house's error tells you something about its neighbor's error, the observations aren't giving you independent information: effective sample size is smaller than n, so standard errors are too small. In time-series data the usual suspect is **serial correlation**, measured by the **Durbin-Watson** statistic: 2 means no correlation between consecutive errors, near 0 means strong positive correlation, near 4 negative. Numeric picture: if each residual is mostly copy-pasted from the previous one, ordering the data by time gives long runs of same-sign residuals and DW near 0.

**Question 5.** (a) Compute DW for the base model's residuals in the file's row order, then re-compute after sorting by `Year_Sold` then `Mo_Sold` (break ties within a month at random — e.g., shuffle first with `.sample(frac=1)` and then use a stable sort — otherwise rows within a month keep the file's order, which is not random). Why do the two differ, and which is the "right" one for a question about *time*? (b) The raw row order is nearly grouped by neighborhood. Check what DW gives sorted by `Neighborhood`. What kind of correlation is being detected? (c) Compute the mean residual per neighborhood. Fit `Sale_Price ~ <base predictors> + C(Neighborhood)` and recompute DW in neighborhood order. Did adding the factor remove the problem? (d) Re-fit the base model with standard errors clustered by neighborhood (`.fit(cov_type='cluster', cov_kwds={'groups': ...})`). How do SEs compare? Which would you trust more for a causal claim about `Gr_Liv_Area`, and what is the caveat with 28 clusters?

**Question 5 (supplementary, not in the book).** Residuals might be correlated in *space* even within a neighborhood. For each house, find its 5 nearest neighbors using `Latitude`/`Longitude` (`sklearn.neighbors.NearestNeighbors`) and average their residuals. Plot or correlate that average with the house's own residual. How large is the correlation, and what does it say about adding a location feature?

*Tools for this question:*
- `durbin_watson(resid_array)` from `statsmodels.stats.stattools`; order the residuals first: `res.resid.loc[order_index]`. Random tiebreak: `df.sample(frac=1, random_state=0).sort_values(["Year_Sold", "Mo_Sold"], kind="stable").index`.
- Clustered SEs: `.fit(cov_type="cluster", cov_kwds={"groups": codes})` with `codes = pd.factorize(df["Neighborhood"])[0]`.
- Spatial extension: `NearestNeighbors(n_neighbors=6).fit(coords).kneighbors(coords)` returns `(distances, indices)`; column 0 is the house itself, so drop it.

_Your interpretation:_

## 6. Partial Residual Plots and Nonlinearity

A residual-vs-fitted plot can look fine even when one predictor is related to y in a curved way, because the other predictors' linear terms can mask it. A **partial residual plot** isolates one predictor: for predictor j, plot `residual + (coef_j * x_j)` against `x_j`. That's "what's left of y after removing everything else's linear contribution", so it should look like a straight line with slope coef_j if the relationship really is linear. A smooth curve through it that bends away from the straight line is the model telling you it needs a nonlinear term. Numeric picture: if the true relationship is a hockey stick (flat then steep) and you fit a line, the partial-residual smooth lies *above* the line at both ends and *below* it in the middle, or the reverse, an S/U shape.

**Question 6.** (a) For the base model use `sm.graphics.plot_ccpr` (one predictor) or `plot_ccpr_grid` (all). Which predictors are well-described by a straight line and which curve? (b) Make your own partial-residual plot for `Year_Built` without the helper (compute `resid + coef * x`, plot it, overlay a lowess smooth from `sm.nonparametric.lowess`). (c) Add `I(Year_Built**2)` to the model. Compare R², RMSE, and the new partial-residual plot. Did the curvature go away? (d) Why do we hold off on interpreting the linear coefficient on `Year_Built` alone once it's in a model with its square? (e) What has the curve got to do with the fact that Ames has a lot of new construction in 2005-2008 and a lot of 1950s houses?

*Tools for this question:*
- `sm.graphics.plot_ccpr(res, "Year_Built")` (one predictor) or `sm.graphics.plot_ccpr_grid(res)` (all).
- `sm.nonparametric.lowess(endog, exog, frac=0.3)` returns an array whose columns are sorted x and smoothed y. Quadratic term: `I(Year_Built**2)` in the formula.

_Your interpretation:_

## 7. Putting It on One Page (interview prep)

**Question 7.** Make a markdown table (in the cell below) with one row per assumption — *linearity of the mean, independent errors, constant variance, normal errors, no perfect multicollinearity, no omitted confounders* — and four columns: **what breaks if it's violated** (coefficients biased? SEs wrong? prediction intervals wrong?), **how you'd detect it**, **a fix**, and **an Ames example you've now seen**. Keep each cell to a short phrase. This is the single most reusable artifact from the chapter for interviews.

_Your table:_

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Outlier vs. leverage vs. influence
*Role lean: DS · interview*

Four houses from Model E (`Sale_Price ~ Gr_Liv_Area + Year_Built + Lot_Area`), with the usual diagnostics:

```text
Model E: Sale_Price~Gr_Liv_Area+Year_Built+Lot_Area   (n = 2930, p = 4 parameters)
Rule-of-thumb flags: |studentized residual| > 3;  leverage > 2p/n = 0.0027;  Cook's D > 4/n = 0.0014

   Gr_Liv_Area  Lot_Area  Year_Built  Sale_Price  stud_resid  leverage_h  cooks_D
A         5642     63887        2008      160000       -11.0      0.0308    0.927
B         2470     12720        2003      615000         6.7      0.0017    0.019
C         2036    215245        1965      375000        -0.9      0.2470    0.072
D         1686     10240        1980      207500         0.0      0.0004    0.000
```

(a) For each of A-D say whether it is an *outlier* (unusual y given x), a *high-leverage point* (unusual x), both, or neither. (b) Which one is truly *influential*, i.e., would change the fitted coefficients noticeably if deleted? Explain why high leverage alone or a large residual alone is not enough. (c) In which direction would you expect the fitted line to be dragged by house A, and what does that do to the `Lot_Area` coefficient? (d) Interview version: "A single data point can swing your regression. Why, and how do you detect it?" 

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Reading a heteroskedasticity diagnosis
*Role lean: DS · Causal · ML*

Same model. Look at how the typical miss changes with the predicted price, and at two ways of computing standard errors:

```text
Residual spread by fitted-value quintile (cheapest predicted homes -> most expensive):
    avg_fitted_price  resid_sd  resid_sd_in_log_model
fq                                                   
0              98431     22441                   0.26
1             138304     24072                   0.19
2             176945     31100                   0.20
3             215836     40991                   0.18
4             274464     78905                   0.27

Coefficient standard errors, same model:
                 coef  classical SE  HC3 robust SE
Gr_Liv_Area    91.685         1.815          4.389
Year_Built   1098.945        29.087         28.949
Lot_Area        0.924         0.113          0.226
```

(a) Describe the pattern in plain words. What would the residuals-versus-fitted plot look like (draw the shape)? (b) Which parts of the output does this problem make *wrong*: the coefficients, their standard errors, the R-squared, the predictions? Be specific about what stays unbiased and what doesn't. (c) The `Gr_Liv_Area` standard error more than doubles with the robust formula while `Year_Built` barely moves. What does that tell you about where in the data the heteroskedasticity is concentrated? (d) Name two fixes and what each buys you (hint: one changes the *model*, one changes only the *standard errors*). (e) After taking logs, the residual spread is roughly even across fitted values. What did the log transform change about the *meaning* of the coefficients?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Non-normal residuals: what actually breaks?
*Role lean: DS · ML · interview*

```text
Residual skewness = 0.53   excess kurtosis = 12.2   (a normal distribution has 0 and 0)

95% prediction intervals from the classical formula, fit on 2,051 training homes, checked on 879 held-out homes:
  held-out homes whose true price fell inside their interval, overall:  96.0%
  ... among the cheapest third:  99.7%
  ... among the middle third:  99.3%
  ... among the priciest third:  89.1%
```

(a) The overall coverage is 96% — close to the 95% target — but that average hides something. Explain the pattern across price tiers using what you know about the residual spread from the previous question. (b) A reviewer says "your residuals aren't normal, so none of your p-values are valid." With n = 2,930, how much of that is true for the *coefficient* p-values, and how much for the *prediction intervals*? (Hint: think about what the CLT does for one but not the other.) (c) What would you do to produce honest prediction intervals here (name two approaches)? (d) If the business question were "give me a range I can promise a seller", which diagnostic from this exam would you care about most?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — Is the independence assumption satisfied?
*Role lean: Causal · DS*

```text
Durbin-Watson of the residuals of Model E (2.0 = no first-order correlation; <2 = neighbors in the ordering resemble each other):
  rows in the order they appear in the file : 1.33
  rows sorted by sale date (ties within a month in random order): 1.95
  rows sorted by Neighborhood               : 1.26

Average residual (actual - fitted) for a few neighborhoods:
                    avg_residual  homes
Neighborhood                           
Landmark                -45065.0      1
Meadow_Village          -38885.0     37
Briardale               -32784.0     30
Northridge_Heights       62015.0    166
Stone_Brook              70223.0     51
Green_Hills              86855.0      2

Standard error of Gr_Liv_Area: classical 1.8   HC3 robust 4.4   clustered by neighborhood (28 clusters) 12.1
```

(a) Why does the Durbin-Watson statistic give such different answers depending on how you sort the rows? What does each of the three numbers tell you about what the data *looks* like (a time effect, or something else)? (b) Based on the neighborhood residual averages, what is the real violation of independence in this dataset, and what does it do to the standard errors? Does it bias the coefficients? (c) The clustered SE for Gr_Liv_Area is much larger than the others. Why is this the "honest" one for a causal question, and what caveat applies when you only have 28 clusters? (d) How would you change the model (not the SEs) to absorb some of this?

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Reading a partial-residual pattern
*Role lean: DS · ML*

```text
Model E again. Homes grouped into 8 equal-sized bins by Year_Built; the average residual (actual - fitted) in each bin:
    avg_year_built  avg_residual  homes
yb                                     
0             1915         12067    372
1             1944          -593    401
2             1959         -4035    340
3             1968        -13458    358
4             1980        -12498    361
5             1997         -8903    398
6             2004           850    376
7             2007         29860    324
```

(a) What does it mean for the average residual to be positive for the oldest homes, negative in the middle, and strongly positive for the newest? What assumption of Model E is this violating? (b) A residual plot against the *fitted values* can look fine even when this pattern exists. Why is plotting residuals against *each predictor* (or a partial-residual plot) the better check? (c) Which coefficient is most distorted by ignoring this: what does the single `Year_Built` slope represent for a model whose true curve looks like this? (d) Propose two specific modifications to fix the shape (you'll implement these in the next notebook), and say which you'd prefer for a model that has to be explained to a non-technical audience.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_

### Exam Q6 — Which assumption protects which claim?
*Role lean: Causal · DS · ML · interview*

Interviewers love a version of this: build a 4-row table in your answer, one row per claim, and for each say which regression assumptions it *needs* and which it *doesn't*. The claims: (1) the coefficient estimates are **unbiased** for the true linear relationship; (2) the **standard errors and p-values** are valid; (3) a **prediction interval** for a new house has the stated coverage; (4) the **point predictions** are good on average. The assumptions: linearity of the mean, errors independent, errors with constant variance, errors normal, no omitted confounders (for a causal reading), no perfect multicollinearity. Then add one sentence on which assumption failures you'd worry about *least* when the only goal is point prediction.

**Your answer (Q6):**

> _write here_

**Review (Q6):**

> _pending — filled in after you answer_

### Exam Q7 — Delete the outliers?
*Role lean: Causal · DS*

```text
Model E with all 2,930 homes vs. after removing the 5 homes with Gr_Liv_Area > 4,000:
               all 2930  without the 5  SE (all)
Intercept   -2132496.18    -2134331.75  56793.62
Gr_Liv_Area       91.68          95.30      1.82
Year_Built      1098.95        1096.29     29.09
Lot_Area           0.92           1.11      0.11

R-squared: 0.667 -> 0.693      Residual std error: 46,144 -> 43,566
The 5 homes have sale prices of 160,000 / 183,850 / 184,750 / 745,000 / 755,000.
```

(a) Is deleting those five homes legitimate? Give an argument for and an argument against. (b) Three of the five sold for under $185k despite being more than twice the size of a typical home, while two sold for over $740k. What would you look up about these homes before deciding what to do with them (columns in the data, or facts about how the data was collected), and what are your options — drop, flag with an indicator, model separately? (c) The `Lot_Area` coefficient moved from 0.92 to 1.11 and R-squared from 0.67 to 0.69. Which of those two changes matters more for the *conclusion* you'd report, and why? (d) What would you write in the methods section of an analysis that dropped them?

**Your answer (Q7):**

> _write here_

**Review (Q7):**

> _pending — filled in after you answer_